# Correlation is not causation

Core, Part 2 (Statistics II and decisions), lesson C10. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** compute and read a correlation coefficient and its limits, name the three rival explanations of a correlation, find a confounder in a dataset, explain Simpson's paradox with a reversal in grouped data, and say what evidence earns causal language.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import linecache
import sys
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def show_error(code, short=False):
    """Run a few lines that fail on purpose and print the error, so Run all keeps going."""
    code = code.strip("\n") + "\n"
    linecache.cache["<cell>"] = (len(code), None, code.splitlines(True), "<cell>")
    try:
        exec(compile(code, "<cell>", "exec"), globals())
    except Exception as error:
        if short:
            print("".join(traceback.format_exception_only(type(error), error)).strip().split("\n")[-1])
        else:
            traceback.print_exception(type(error), error, error.__traceback__.tb_next, file=sys.stdout)

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, MAGENTA, GOLD, GREEN = "#0072b2", "#d6479a", "#ad9100", "#00a07a"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, MAGENTA, GOLD, GREEN]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
users = pd.read_csv(BASE + "users.csv", parse_dates=["signup_date"])
events = pd.read_csv(BASE + "events.csv", parse_dates=["event_ts"], usecols=["user_id", "event_ts", "event_name", "platform"])
assignments = pd.read_csv(BASE + "experiment_assignments.csv")
from scipy import stats

# Known answer from the dataset README: 21,841 users, 177,379 events and 15,868 experiment rows.
assert (len(users), len(events), len(assignments)) == (21841, 177379, 15868)
print("Data loaded: 21,841 users, 177,379 events, 15,868 experiment rows.")
# One row for each person who signed up by June 16, so that everyone has 14 days of follow-up. `platform` is the platform of the signup event.
signup = events[events["event_name"] == "signup_complete"][["user_id", "event_ts", "platform"]].rename(columns={"event_ts": "signup_ts"})
people = users[users["signup_date"] <= "2026-06-16"].merge(signup, on="user_id", how="left", validate="one_to_one")
ev = events.merge(people[["user_id", "signup_ts"]], on="user_id")  # the events of these people only
ev["days"] = (ev["event_ts"] - ev["signup_ts"]).dt.total_seconds() / 86400  # days since the person's signup
# bought: a purchase within 14 days of signup. events7: visits (session_start) and expenses (add_expense) in the first 7 days, counted as the rows come.
bought_ids = ev.loc[(ev["event_name"] == "purchase") & (ev["days"] <= 14), "user_id"].unique()
week1 = ev[ev["event_name"].isin(["session_start", "add_expense"]) & (ev["days"] >= 0) & (ev["days"] <= 7)]
people["bought"] = people["user_id"].isin(bought_ids).astype(int)
people["events7"] = people["user_id"].map(week1.groupby("user_id").size()).fillna(0).astype(int)
people["engaged"] = (people["events7"] >= 5).astype(int)  # engaged: 5 or more events in the first week
assert len(people) == 20177
print("20,177 people with 14 days of follow-up. `people` has bought (0 or 1), events7 and engaged (0 or 1).")

## The question

In [ ]:
summary = people.groupby("engaged")["bought"].agg(people="count", bought_pct="mean")
summary["bought_pct"] = (summary["bought_pct"] * 100).round(1)
print(summary)

## Unit 1: One number for a scatter

**Predict first.** r runs from −1 to 1. Will the correlation between a person's events in the first week and buying be close to 0, around 0.5 or close to 1?

In [ ]:
x, y = people["events7"], people["bought"]
print(round(x.corr(y), 3))
dx, dy = x - x.mean(), y - y.mean()
print(round((dx * dy).sum(), 2), round((dx ** 2).sum(), 2), round((dy ** 2).sum(), 2))
print(round(x.mean(), 2), round(y.mean(), 2))

**Predict first.** Four small pictures, A to D, have 12 points each. A is a rising cloud, B a curve that rises and then falls, C a straight line with one stray point, D eleven points in a column and one far to the side. Which one has the highest r?

In [ ]:
shapes = {
    "A": (list(range(1, 13)), [4.5, 3.1, 4.7, 5.9, 3.8, 6.2, 4.9, 6.8, 7.8, 5.9, 7.8, 7.2]),
    "B": (list(range(1, 13)), [1.0, 3.4, 5.5, 7.2, 8.6, 9.6, 10.3, 10.7, 10.7, 10.3, 9.7, 8.7]),
    "C": (list(range(1, 13)), [3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 5.2]),
    "D": ([3] * 11 + [12], [4.1, 4.9, 5.6, 6.2, 5.1, 6.9, 4.5, 5.9, 5.3, 6.5, 5.7, 9.2]),
}
for name, (a, b) in shapes.items():
    print(name, f"{np.corrcoef(a, b)[0, 1]:.2f}")

## Unit 2: Three rival explanations

**Predict first.** People who bought within 14 days made 7,898 visits and expenses in their first week. What share of them happened after the purchase: a few, about half or most?

In [ ]:
first_buy = ev[(ev["event_name"] == "purchase") & (ev["days"] <= 14)].groupby("user_id", as_index=False)["event_ts"].min().rename(columns={"event_ts": "buy_ts"})
buyers_week1 = week1.merge(first_buy, on="user_id")  # the first-week events of the people who bought
print(len(buyers_week1), round((buyers_week1["event_ts"] > buyers_week1["buy_ts"]).mean(), 3))

**Predict first.** Cut the referral people into blocks of 30, in the order they signed up, and compute r between events7 and bought in every block. What will the largest r be, and in how many blocks will |r| reach 0.3 or more?

In [ ]:
referral = people[people["channel"] == "referral"].sort_values("user_id")
block = np.arange(len(referral)) // 30  # blocks of 30 people, in signup order
r_by_block = referral.groupby(block)[["events7", "bought"]].apply(lambda d: d["events7"].corr(d["bought"]) if len(d) == 30 else np.nan).dropna()
print(len(r_by_block), round(r_by_block.min(), 2), round(r_by_block.max(), 2), int((r_by_block.abs() >= 0.3).sum()))

## Unit 3: Find the confounder

**Predict first.** Which channel has the largest share of engaged people, and which has the highest share who bought? And which has the smallest of each?

In [ ]:
by_channel = people.groupby("channel").agg(people=("user_id", "count"), engaged_pct=("engaged", "mean"), bought_pct=("bought", "mean"))
by_channel[["engaged_pct", "bought_pct"]] = (by_channel[["engaged_pct", "bought_pct"]] * 100).round(1)
print(by_channel.sort_values("bought_pct", ascending=False))

**Predict first.** Inside each channel, will engaged people still buy about 5 points more often than the others? And what will r be inside each channel?

In [ ]:
rates = people.pivot_table(index="channel", columns="engaged", values="bought", aggfunc="mean").mul(100).round(1)
rates.columns = ["not engaged", "engaged"]
rates["gap"] = (rates["engaged"] - rates["not engaged"]).round(1)
print(rates.sort_values("not engaged", ascending=False))
within = people.groupby("channel")[["events7", "bought"]].apply(lambda d: d["events7"].corr(d["bought"])).round(3)
print(within.to_dict())

## Unit 4: A reversal inside the groups

**Predict first.** Which platform of the signup converts best overall: web, iOS or Android?

In [ ]:
print(people.groupby("platform")["bought"].mean().mul(100).round(1).to_dict())

**Predict first.** Inside each channel that has web people, who converts better, web or iOS? And how many of the four gaps are larger than chance would give?

In [ ]:
cells = people.pivot_table(index="channel", columns="platform", values="bought", aggfunc="mean").mul(100).round(1)
print(cells[["web", "ios"]])
for c in ["organic_search", "paid_search", "paid_social", "referral"]:
    d = people[people["channel"] == c]
    table = pd.crosstab(d["platform"], d["bought"]).loc[["web", "ios"]]
    print(c, table.sum(axis=1).tolist(), round(stats.chi2_contingency(table, correction=False)[1], 4))

**Predict first.** Which two channels hold most of the web people, and which two hold most of the iOS people?

In [ ]:
mix = pd.crosstab(people["platform"], people["channel"], normalize="index").mul(100).round(1)
print(mix.loc[["web", "ios"]])

## Unit 5: What earns causal language

**Predict first.** In the paywall_v2 test, a coin put each new person in the control or the variant group. Will the channel mix of the two groups differ by more than a point or two?

In [ ]:
paywall = assignments[assignments["experiment"] == "paywall_v2"].merge(people[["user_id", "channel"]], on="user_id", validate="one_to_one")
print(pd.crosstab(paywall["variant"], paywall["channel"], normalize="index").mul(100).round(1))
print(paywall.groupby("variant")["converted"].mean().mul(100).round(1).to_dict())

## Finish it

### Exercise 1: Finish it

**Task:** Is `region` a confounder of the same link? Take the people of Europe and compare the share who bought among the engaged and among the others. Put the gap, engaged minus others, in percentage points rounded to 1 decimal in `answer_1`.

**Expected:** `answer_1` is a number in percentage points.

In [ ]:
europe = people[people["region"] == "europe"]  # given: the people of Europe
# the share who bought among the engaged and among the others, in percent; set answer_1 to the gap in points
answer_1 = None  # replace None

In [ ]:
_ = check("Exercise 1", "answer_1", 5.2, traps={5.5: "That is the gap for all 20,177 people. Take only the people of Europe first.", 0.052: "That is a share between 0 and 1. The task asks for percentage points: multiply by 100."})

<details>
<summary>Hint</summary>

Group `europe` by `engaged` and take the mean of `bought`; the gap is the difference of two means.

</details>

<details>
<summary>Solution</summary>

```python

europe = people[people["region"] == "europe"]
shares = europe.groupby("engaged")["bought"].mean() * 100
answer_1 = round(float(shares[1] - shares[0]), 1)

```

5.2 points, close to the 5.5 of all 20,177 people: knowing a person's region does not explain the gap, because region is not tied to both columns (the dataset README says it changes nothing). A confounder must be tied to both sides.

</details>

## On your own

### Exercise 2: On your own

**Task:** What conversion would iOS people have if they came through the channels in the same shares as web people? Before you start, say in one sentence what the result looks like: one percentage between 0 and 100. Weight the iOS conversion of each channel by the share of all web people who came through that channel, add the four products, and put the percentage rounded to 1 decimal in `answer_2`. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a percentage, a number between 0 and 100.

In [ ]:
_ = check("Exercise 2", "answer_2", 19.4, traps={12.5: "That is the plain iOS rate, with iOS's own channel mix. Weight each channel by the web share, not the iOS share.", 16.3: "That is the web rate. The task asks for iOS conversion at the web channel mix."})

<details>
<summary>Hint</summary>

Two pieces: the share of web people in each channel (`value_counts(normalize=True)`), and the iOS conversion of each channel (a groupby). Multiply them channel by channel.

</details>

<details>
<summary>Solution</summary>

```python

web_mix = people[people["platform"] == "web"]["channel"].value_counts(normalize=True)
ios_rates = people[people["platform"] == "ios"].groupby("channel")["bought"].mean()
answer_2 = round(float((web_mix * ios_rates.loc[web_mix.index]).sum() * 100), 1)

```

19.4%, against 16.3% for web people: with the same channel mix, iOS is ahead, which reverses the overall order. The four web channels are the ones with web people; `app_store` has none, so it drops out of the weights.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague wants the correlation between being engaged and buying, and computes it on the five channels: the share engaged and the share who bought, one pair for each channel. Fix the last line so that the correlation is computed over the 20,177 people, `engaged` against `bought`, and set `answer_3` to it rounded to 3 decimals.

**Expected:** `answer_3` is a number between −1 and 1.

In [ ]:
by_channel_means = people.groupby("channel")[["engaged", "bought"]].mean()  # one row for each channel: five rows
answer_3 = round(float(by_channel_means["engaged"].corr(by_channel_means["bought"])), 3)  # the colleague's number

In [ ]:
_ = check("Exercise 3", "answer_3", 0.065, traps={0.991: "That is the correlation of five channel averages. Each row of the correlation must be one person, not one channel."})

<details>
<summary>Hint</summary>

The colleague's table has five rows. How many rows does the question need?

</details>

<details>
<summary>Solution</summary>

```python

answer_3 = round(float(people["engaged"].corr(people["bought"])), 3)

```

0.065 over the 20,177 people. The correlation of the five channel averages is 0.991, because averages hide the spread inside each channel: it says something about channels, not about people.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")